# Implementation Notes

In [1]:
# encode geons
# maybe the landmark geon is the one that is most unique compared to base shape (i.e. super long skinny rectangular prism is more unique than short/stout rectangular prism)
# geon hrrs for type, size, aspect ratio, degree of curvature, tapering.
# type, size, aspect ratio is base(?). curvature/tapering is details --> which isnt present in shepard + metzler experiments :P

# ok. now how to do spatial relations between two geons. lol
# 
'''
- research granularity of rotation (?)
- hutttenlocher


ok. i think best way will be to have a data structure with geons and relations. holographic vectors are used to decribe them 

---
ENCODING 3D OBJECTS
rhcd --> 
ssps --> semantic somethings
- josh and encoding objects 
- look for data that has response time/error --> things that get more complicated w/ shepard metzler shapes
---

'''



'''
determine axis of rotation

- determine approximate center of overall shape
- determine based on center of shape if landmark shape must be moved left/right, up/down, backwards/forwards (distance will determine x,y,z coords of axis accordingly)
    - ex: which one is further forward? which one is higher? which one is closer to the left?
    - difference value for each of these questions rated 0-1. these difference values determine vector for axis of rotation (will go through approximate center)
- determine approximate axis of rotation

- will be updated during rotation to be more accurate (function approximation to find correct one?)

determine direction of rotation

- determine center of overall shape
- whichever direction makes more sense based on left/right, up/down, backwards/forwards differences between landmark geons 
- end up with a vector with a -1 (negative movement) or 1 (positive movement) for each direction
'''

'''
COMPLEXITY FACTORS TO KEEP IN MIND

- picture plane rotation faster than depth rotation (i think picture plane difference measures is prob more accurate)
- combined rotation axes are slower than base axes (like x, y, z alone)
- objects with occlusion slower (this would be interesting to test when considering landmarks)

'''

'\nCOMPLEXITY FACTORS TO KEEP IN MIND\n\n- picture plane rotation faster than depth rotation (i think picture plane difference measures is prob more accurate)\n- combined rotation axes are slower than base axes (like x, y, z alone)\n- objects with occlusion slower (this would be interesting to test when considering landmarks)\n\n'

# Imports

In [2]:
import numpy as np
from scipy.spatial.transform import Rotation as rotate
from geon_relations_v2 import Object, RectangularPrism, find_axis_of_rotation, find_center_point_LWLC
import plotly.graph_objects as go
import plotly.express as px
import plotly.offline as pyo
from plotly.subplots import make_subplots
import math, copy

# Initialize Plotly for offline mode in Jupyter Notebook
pyo.init_notebook_mode(connected=True)

# Plotting Functions

In [3]:
def add_point(fig, coords, name, colour='green'):
        
    fig.add_trace(*create_data_from_point(coords, name, colour))                # create_data returns a list, the * unpacks it

def add_landmarks(fig, landmarks, name, colour='orange'):

    showlegend = True

    for landmark in landmarks:
        fig.add_trace(go.Scatter3d(
                    x=[landmark[0]],
                    y=[landmark[1]],
                    z=[landmark[2]],
                    mode='markers',
                    marker=dict(
                        size=5,
                        color=colour
                    ),
                    name=name,
                    legendgroup=name,
                    showlegend=showlegend
        ))

        showlegend = False              # false after first loop!

def add_axis(fig, axis, colour='green', scale=1):

    fig.add_trace(*create_data_from_axis(axis, colour, scale))          # create_data returns a list, the * unpacks it

def add_object(fig, object, name, colour='red', row=None, col=None):

    vector_start = object.start_coords.copy()
    vector_end = object.start_coords.copy()
    showlegend = True

    for geon in object.geons:
        direction_vector = geon.direction * geon.length
        vector_start = vector_end.copy()
        vector_end += direction_vector
        
        if row == None or col == None:
            fig.add_trace(go.Scatter3d(
                x=[vector_start[0], vector_end[0]],
                y=[vector_start[1], vector_end[1]],
                z=[vector_start[2], vector_end[2]],
                mode='lines',
                line=dict(color=colour, width=5),
                name=name,
                legendgroup=name,
                showlegend=showlegend
            ))
        else:
            fig.add_trace(go.Scatter3d(
                x=[vector_start[0], vector_end[0]],
                y=[vector_start[1], vector_end[1]],
                z=[vector_start[2], vector_end[2]],
                mode='lines',
                line=dict(color=colour, width=5),
                name=name,
                legendgroup=name,
                showlegend=showlegend
                ),
            row=row,
            col=col
            )

        showlegend=False            # false after first loop!

def create_data_from_point(coords, name, colour='green'):

    return [go.Scatter3d(
        x=[coords[0]],
        y=[coords[1]],
        z=[coords[2]],
        mode='markers',
        marker=dict(
            size=5,
            color=colour
        ),
        name=name,
    )]

def create_data_from_landmarks(landmarks, name, colour='orange'):

    data = []
    showlegend = True

    for landmark in landmarks:
        data.append(go.Scatter3d(
                    x=[landmark[0]],
                    y=[landmark[1]],
                    z=[landmark[2]],
                    mode='markers',
                    marker=dict(
                        size=5,
                        color=colour
                    ),
                    name=name,
                    legendgroup=name,
                    showlegend=showlegend
        ))

        showlegend = False              # false after first loop!

    return data

def create_data_from_axis(axis, colour='green', scale=1):
    axis_start = axis * -scale
    axis_end = axis * scale

    return [go.Scatter3d(
        x=[axis_start[0], axis_end[0]],
        y=[axis_start[1], axis_end[1]],
        z=[axis_start[2], axis_end[2]],
        mode='lines',
        line=dict(color=colour, width=5, dash='dot'),
        name="Axis of Rotation"
    )]

def create_data_from_object(object, name, colour='blue'):

    data = []
    showlegend = True

    vector_start = object.start_coords.copy()
    vector_end = object.start_coords.copy()

    for geon in object.geons:
        direction_vector = geon.direction * geon.length
        vector_start = vector_end.copy()
        vector_end += direction_vector
        
        data.append(go.Scatter3d(
            x=[vector_start[0], vector_end[0]],
            y=[vector_start[1], vector_end[1]],
            z=[vector_start[2], vector_end[2]],
            mode='lines',
            line=dict(color=colour, width=5),
            name=name,
            legendgroup=name,
            showlegend=showlegend
        ))

        showlegend=False            # false after first loop!

    return data

def add_frame(frame_array, object, object_name, landmark_name=None, axis_of_rotation=None, object_colour='blue', landmark_colour='purple', axis_colour='green', axis_scale=1):

    data = create_data_from_object(object, object_name, object_colour)

    if landmark_name is not None:
        data = data + create_data_from_landmarks(object.get_landmark_endpoints(), landmark_name, landmark_colour)
    if axis_of_rotation is not None:
        data = data + create_data_from_axis(axis_of_rotation, axis_colour, axis_scale)

    traces = [i for i in range(len(data))]
    name = f'frame{len(frame_array)}'

    frame_array.append(go.Frame(data=data, traces=traces, name=name))

# Encode Objects (Geons/Relations/HRRs)

This is the case I'll model:

![image](test.jpg)

*150 degree diff in pic

In [4]:
# r = rotate.from_euler('z', 60, degrees=True)                                  # 60 deg rotation around z-axis
r = rotate.from_rotvec([0, 0, np.deg2rad(150)])                               # 150 deg rotation around z-axis
# r = rotate.from_rotvec([0, 0, np.deg2rad(-219)])                              # -219 deg rotation around z-axis
# r = rotate.from_rotvec([0, np.deg2rad(180), 0])                               # 180 deg rotation around y-axis
# r = rotate.from_rotvec([0, np.deg2rad(220), 0])                               # 220 deg rotation around y-axis
# r = rotate.from_rotvec([0, np.deg2rad(-60), 0])                               # -60 deg rotation around y-axis
# r = rotate.from_rotvec([np.deg2rad(150), 0, 0])                               # 150 deg rotation around x-axis
# r = rotate.from_rotvec([0, np.deg2rad(60), np.deg2rad(30)])                   # 60 deg rotation around y-axis, 30 deg rotation around z-axis
# r = rotate.from_rotvec([0, np.deg2rad(180), np.deg2rad(180)])                 # 180 deg rotation around y-axis, 180 deg rotation around z-axis
# r = rotate.from_rotvec([np.deg2rad(150), np.deg2rad(60), np.deg2rad(90)])       # crazy rotation :o

# x: right is positive, y: further away is positive, z: up is positive

# create original geons
g1 = RectangularPrism(2, np.array([-1, -1, 0]))             # 45 deg angle front left, no z info
g2 = RectangularPrism(3, np.array([0, 0, -1]))              # down
# g1 = RectangularPrism(2, np.array([0.02, 0.0, 1.0]))      # 45 deg angle front left, no z info
# g2 = RectangularPrism(3, np.array([-0.02, 0.0, 2.0]))     # down
g3 = RectangularPrism(2, np.array([1, 1, 0]))               # 45 deg angle back right, no z info
g4 = RectangularPrism(1, np.array([1, -1, 0]))              # 45 deg angle front right, no z info

# create original object and relations
original_object = Object(
# target_object = Object(
    geons = [g1,g2,g3,g4],
    landmark_geon_index = 0                    # i.e. landmark is g1
)

# # create target geons (same as original, but with rotation applied)
# g1 = RectangularPrism(2, r.apply(np.array([-1, -1, 0])))
# g2 = RectangularPrism(3, r.apply(np.array([0, 0, -1])))
# # g1 = RectangularPrism(2, r.apply(np.array([0.02, 0.0, 1.0])))
# # g2 = RectangularPrism(3, r.apply(np.array([-0.02, 0.0, 2.0])))
# g3 = RectangularPrism(2, r.apply(np.array([1, 1, 0])))
# g4 = RectangularPrism(1, r.apply(np.array([1, -1, 0])))

# MIRRORED target object
g1 = RectangularPrism(2, r.apply(np.array([-1, -1, 0])))
g2 = RectangularPrism(3, r.apply(np.array([0, 0, -1])))
g3 = RectangularPrism(2, r.apply(np.array([1, 1, 0])))
g4 = RectangularPrism(1, r.apply(np.array([-1, 1, 0])))

target_object = Object(
# original_object = Object(
    geons = [g1,g2,g3,g4],
    landmark_geon_index = 0                    # i.e. landmark is g1
)

In [5]:
# Create graph
fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'scene'}, {'type': 'scene'}]], subplot_titles=("Original Object", "Target Object"))

add_object(fig, original_object, "Original Object", colour='blue', row=1, col=1)
add_object(fig, target_object, "Target Object", colour='red', row=1, col=2)

fig.update_layout(title='3D Vector Visualization', showlegend=False)

# Calculate Axis of Rotation

In [6]:
center_point = np.array([0,0,0])
axis_of_rotation, direction, prev_angle = find_axis_of_rotation(original_object, target_object, center_coords=center_point)
axis_of_rotation = axis_of_rotation - np.array([0,0,0.3])
print(direction)

[0. 0. 1.]
1.0


In [7]:
# Create graph
axis_fig = go.Figure(data=[])

print(original_object.get_landmark_endpoints())

add_object(axis_fig, original_object, "Original Object", colour='blue')
add_object(axis_fig, target_object, "Target Object", colour='red')

add_landmarks(axis_fig, original_object.get_landmark_endpoints(), "Original Landmarks", colour='purple')
add_landmarks(axis_fig, target_object.get_landmark_endpoints(), "Target Landmarks", colour='orange')
add_point(axis_fig, center_point, "Rotation Point", colour='green')

add_axis(axis_fig, axis_of_rotation, scale=2)

axis_fig.update_layout(title='3D Vector Visualization')

[[ 1.41421356  1.41421356  0.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.         -1.        ]]


# Perform Rotation

In [8]:
# functions
def cosine_similarity(vec_a, vec_b):
    return np.dot(vec_a, vec_b) / (np.linalg.norm(vec_a) * np.linalg.norm(vec_b))

def sigmoid(x, min, max):
    return (1 / (1 + math.exp(-x))) * (max - min + 1) + min

def overall_landmark_distance(landmarks1, landmarks2):
    total_distance = 0
    for index in range(len(landmarks1)):
        landmark_diff = landmarks2[index] - landmarks1[index]
        total_distance += np.linalg.norm(landmark_diff)

    return total_distance

def frame_args(duration):
    return {
            "frame": {"duration": duration},
            "mode": "immediate",
            "fromcurrent": True,
            "transition": {"duration": duration, "easing": "linear"},
            }

# define graph w/ axis animation
size = 4
axis_animation_fig = go.Figure(
    data =  create_data_from_object(original_object, "Original Object", colour='blue')
            + create_data_from_landmarks(original_object.get_landmark_endpoints(), "Original Landmark", colour='purple')
            + create_data_from_axis(axis_of_rotation, scale=2)
            + create_data_from_object(target_object, "Target Object", colour='red')
            + create_data_from_landmarks(target_object.get_landmark_endpoints(), "Target Landmark", colour='orange')
            + create_data_from_point(center_point, "Rotation Point", colour='green'),
    layout=go.Layout(
        scene=dict(
            xaxis=dict(range=[-size, size], autorange=False),
            yaxis=dict(range=[-size, size], autorange=False),
            zaxis=dict(range=[-size, size], autorange=False),
            aspectmode='cube'
        ),
        title="Animated Rotation",
        updatemenus=[dict(
            type="buttons",
            buttons=[dict(label="Play", method="animate", args=[None, frame_args(50)])]
        )]
    )
)
axis_animation = []

# define graph w/ overlap animation
original_centerpoint_vec = find_center_point_LWLC(original_object)
target_centerpoint_vec = find_center_point_LWLC(target_object)
copy_og_obj = copy.deepcopy(original_object)
copy_tar_obj = copy.deepcopy(target_object)

copy_og_obj.update_start_coords(copy_og_obj.start_coords - original_centerpoint_vec)
copy_tar_obj.update_start_coords(copy_tar_obj.start_coords - target_centerpoint_vec)

size = 4
overlap_animation_fig = go.Figure(
    data =  create_data_from_object(copy_og_obj, "Original Object", colour='blue')
            + create_data_from_object(copy_tar_obj, "Target Object", colour='red'),
    layout=go.Layout(
        scene=dict(
            xaxis=dict(range=[-size, size], autorange=False),
            yaxis=dict(range=[-size, size], autorange=False),
            zaxis=dict(range=[-size, size], autorange=False),
            aspectmode='cube'
        ),
        title="Animated Rotation",
        updatemenus=[dict(
            type="buttons",
            buttons=[dict(label="Play", method="animate", args=[None, frame_args(50)])]
        )]
    )
)
overlap_animation = []

# get landmark vectors
original_landmark_geon_vector = original_object.get_landmark_geon().get_vector()
target_landmark_geon_vector = target_object.get_landmark_geon().get_vector()
# original_landmark_vector
# target_landmark_vector

max_step_size = 5                       # TODO: change to a range that is affected by distance to landmark
min_step_size = 2

loop_count = 0
theta = None

landmark_angle_threshold = 0.975
landmark_distance_threshold = 0.5

smaller_landmark_angle_threshold = 0.8
smaller_landmark_distance_threshold = 2

step_size = max_step_size

while cosine_similarity(original_landmark_geon_vector, target_landmark_geon_vector) < landmark_angle_threshold or overall_landmark_distance(original_object.get_landmark_endpoints(), target_object.get_landmark_endpoints()) > landmark_distance_threshold:     # checking cosine similarity and distance between landmarks

    # # find best axis of rotation, create rotation function
    axis_of_rotation, direction, prev_angle = find_axis_of_rotation(original_object, target_object, center_coords=center_point, prev_axis=axis_of_rotation, prev_direction=direction, prev_angles=prev_angle)
    # print(axis_of_rotation)
    print(direction)
    r = rotate.from_rotvec(direction * np.deg2rad(step_size) * axis_of_rotation)        # quaternion representing step size rotation around calculated axis

    # apply rotation to original object
    original_object.rotate(r)

    # update original_landmark_vector
    original_landmark_geon_vector = original_object.get_landmark_geon().get_vector()

    # add animation frame to graph
    add_frame(axis_animation, original_object, object_name="Original Object", landmark_name="Original Landmarks", axis_of_rotation=axis_of_rotation, object_colour='blue', landmark_colour='purple', axis_colour='green', axis_scale=2)

    original_centerpoint_vec = find_center_point_LWLC(original_object)
    copy_og_obj = copy.deepcopy(original_object)
    copy_og_obj.update_start_coords(copy_og_obj.start_coords - original_centerpoint_vec)
    add_frame(overlap_animation, copy_og_obj, object_name="Original Object", object_colour='blue', axis_scale=2)

    loop_count += 1

    if loop_count > 100:
        break

    # if overall_landmark_distance(original_object.get_landmark_endpoints(), target_object.get_landmark_endpoints()) < smaller_landmark_distance_threshold:
    #     step_size = min_step_size

    # step_size = min(max(np.round(np.sqrt((max_step_size-min_step_size+1) * overall_landmark_distance(original_object.get_landmark_endpoints(), target_object.get_landmark_endpoints()))), min_step_size), step_size)
    print(step_size)

axis_animation_fig.frames = axis_animation
axis_animation_fig.show()
overlap_animation_fig.frames = overlap_animation
overlap_animation_fig.show()

[0. 0. 1.]
1.0
5
[0. 0. 1.]
1.0
5
[0. 0. 1.]
1.0
5
[0. 0. 1.]
1.0
5
[0. 0. 1.]
1.0
5
[0. 0. 1.]
1.0
5
[0. 0. 1.]
1.0
5
[0. 0. 1.]
1.0
5
[0. 0. 1.]
1.0
5
[0. 0. 1.]
1.0
5
[0. 0. 1.]
1.0
5
[0. 0. 1.]
1.0
5
[0. 0. 1.]
1.0
5
[0. 0. 1.]
1.0
5
[0. 0. 1.]
1.0
5
[0. 0. 1.]
1.0
5
[0. 0. 1.]
1.0
5
[0. 0. 1.]
1.0
5
[0. 0. 1.]
1.0
5
[0. 0. 1.]
1.0
5
[0. 0. 1.]
1.0
5
[0. 0. 1.]
1.0
5
[0. 0. 1.]
1.0
5
[0. 0. 1.]
1.0
5
[0. 0. 1.]
1.0
5
[0. 0. 1.]
1.0
5
[0. 0. 1.]
1.0
5
[0. 0. 1.]
1.0
5
